# Spotify Music Recommendation System
## Notebook 04 — Feature Engineering

**Purpose:** Transform cleaned audio features into a normalised 14-dimensional vector that will be used for cosine-similarity-based recommendation.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Cleaned Data](#2-load-cleaned-data)
3. [Feature Plan](#3-feature-plan)
4. [Step 1 — Log-Transform Skewed Features](#4-step-1--log-transform-skewed-features)
5. [Step 2 — Scale All Features to 0–1](#5-step-2--scale-all-features-to-01)
6. [Step 3 — Cyclic Encoding for Key](#6-step-3--cyclic-encoding-for-key)
7. [Step 4 — Binary Features](#7-step-4--binary-features)
8. [Assemble & Validate Feature Matrix](#8-assemble--validate-feature-matrix)
9. [Save Outputs](#9-save-outputs)
10. [Summary](#10-summary)

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR   = Path('../data/processed')
OUTPUT_DIR = Path('../data/processed')

## 2. Load Cleaned Data

In [ ]:
df = pd.read_csv(DATA_DIR / 'data_clean.csv')
print(f'Loaded: {df.shape[0]:,} tracks, {df.shape[1]} columns')
df[['id', 'name', 'artists_clean', 'year', 'popularity', 'energy', 'valence']].head(3)

## 3. Feature Plan

We use **14 features** for the recommendation engine:

| Feature | Raw Column | Transform | Why |
|---|---|---|---|
| valence | `valence` | MinMax | Already 0–1 |
| acousticness | `acousticness` | MinMax | Already 0–1 |
| danceability | `danceability` | MinMax | Already 0–1 |
| energy | `energy` | MinMax | Already 0–1 |
| loudness | `loudness` | MinMax | Range −60 to 0 dB |
| log_instrumentalness | `instrumentalness` | log1p + MinMax | Right-skewed |
| log_speechiness | `speechiness` | log1p + MinMax | Right-skewed |
| log_liveness | `liveness` | log1p + MinMax | Right-skewed |
| tempo_norm | `tempo` | z-score + MinMax | Not bounded |
| key_sin | `key` | sin(2π·key/12) | Circular feature |
| key_cos | `key` | cos(2π·key/12) | Circular feature |
| mode | `mode` | as-is | Binary 0/1 |
| explicit_int | `explicit` | int cast | Binary 0/1 |
| popularity_norm | `popularity` | MinMax | 0–100 scale |

## 4. Step 1 — Log-Transform Skewed Features

`instrumentalness`, `speechiness`, and `liveness` have most values near 0. `log1p(x)` spreads these values out so the recommender is not dominated by a few extreme tracks.

In [ ]:
SKEWED = ['instrumentalness', 'speechiness', 'liveness']

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for i, feat in enumerate(SKEWED):
    axes[0, i].hist(df[feat], bins=50, color='salmon', edgecolor='none')
    axes[0, i].set_title(f'{feat} — raw', fontweight='bold')
    axes[1, i].hist(np.log1p(df[feat]), bins=50, color='steelblue', edgecolor='none')
    axes[1, i].set_title(f'{feat} — log1p', fontweight='bold')
plt.suptitle('Effect of log1p: spreads out right-skewed data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

for feat in SKEWED:
    df[f'log_{feat}'] = np.log1p(df[feat])
    before, after = df[feat].skew(), df[f'log_{feat}'].skew()
    print(f'{feat}: skew {before:.2f} -> {after:.2f}')

## 5. Step 2 — Scale All Features to 0–1

MinMaxScaler maps each feature to the 0–1 range. This ensures no single feature dominates the cosine similarity calculation just because it has a larger raw value range.

In [ ]:
MINMAX_COLS = ['valence', 'acousticness', 'danceability', 'energy', 'loudness',
               'log_instrumentalness', 'log_speechiness', 'log_liveness']

mm = MinMaxScaler()
scaled = mm.fit_transform(df[MINMAX_COLS])
df_feat = pd.DataFrame(scaled, columns=MINMAX_COLS, index=df.index)

# Tempo: z-score first (removes BPM units), then MinMax to 0-1
tempo_z = StandardScaler().fit_transform(df[['tempo']])
df_feat['tempo_norm'] = MinMaxScaler().fit_transform(tempo_z)

print('All scaled to 0-1. Check min/max:')
print(df_feat.describe().loc[['min', 'max']].round(4).to_string())

## 6. Step 3 — Cyclic Encoding for Key

Musical keys are circular: after B (11) comes C (0) again. If we just use the raw number, the model thinks B and C are far apart (11 vs 0), but musically they are adjacent. Sin/cos encoding fixes this.

In [ ]:
df_feat['key_sin'] = np.sin(2 * np.pi * df['key'] / 12)
df_feat['key_cos'] = np.cos(2 * np.pi * df['key'] / 12)

# Visual proof: keys arranged in a circle
key_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
angles = [2 * np.pi * k / 12 for k in range(12)]
kx = np.sin(angles)
ky = np.cos(angles)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(kx, ky, s=120, color='steelblue', zorder=3)
for label, x, y in zip(key_labels, kx, ky):
    ax.annotate(label, (x, y), fontsize=10, ha='center', va='bottom',
                fontweight='bold', xytext=(0, 8), textcoords='offset points')
ax.set_title('Keys are circular: C and B are neighbours', fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Step 4 — Binary Features

In [ ]:
df_feat['mode']         = df['mode'].astype(int)
df_feat['explicit_int'] = df['explicit'].astype(int)
df_feat['popularity_norm'] = MinMaxScaler().fit_transform(df[['popularity']])

print('Binary and popularity features added.')
print('explicit_int values:', df_feat['explicit_int'].value_counts().to_dict())
print('mode values        :', df_feat['mode'].value_counts().to_dict())

## 8. Assemble & Validate Feature Matrix

In [ ]:
FEATURE_COLUMNS = [
    'valence', 'acousticness', 'danceability', 'energy', 'loudness',
    'log_instrumentalness', 'log_speechiness', 'log_liveness',
    'tempo_norm',
    'key_sin', 'key_cos',
    'mode', 'explicit_int',
    'popularity_norm',
]

feature_matrix = df_feat[FEATURE_COLUMNS].copy()

# Track index — what we show to the user for each recommendation
track_index = df[['id', 'name', 'artists_clean', 'year', 'popularity']].copy()
track_index = track_index.rename(columns={'artists_clean': 'artists'})

print(f'Feature matrix : {feature_matrix.shape}  ({len(FEATURE_COLUMNS)} features per track)')
print(f'Track index    : {track_index.shape}')
print()

# Sanity check
nan_count = feature_matrix.isnull().sum().sum()
inf_count = np.isinf(feature_matrix.values).sum()
print(f'NaN values: {nan_count}  (should be 0)')
print(f'Inf values: {inf_count}  (should be 0)')

## 9. Save Outputs

In [ ]:
feature_matrix.to_csv(OUTPUT_DIR / 'feature_matrix.csv', index=False)
track_index.to_csv(OUTPUT_DIR / 'track_index.csv', index=False)

with open(OUTPUT_DIR / 'feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print('Saved:')
print(f'  feature_matrix.csv   {feature_matrix.shape}')
print(f'  track_index.csv      {track_index.shape}')
print(f'  feature_columns.json {len(FEATURE_COLUMNS)} features')

## 10. Summary

The feature vector has **14 dimensions**. Every value is in [−1, 1] — most are in [0, 1], only `key_sin` and `key_cos` can be negative.

| Group | Features | Transform |
|---|---|---|
| Audio (continuous) | valence, acousticness, danceability, energy, loudness | MinMax |
| Audio (skewed) | instrumentalness, speechiness, liveness | log1p + MinMax |
| Tempo | tempo_norm | z-score + MinMax |
| Key | key_sin, key_cos | Cyclic sin/cos |
| Binary | mode, explicit_int | 0 / 1 |
| Popularity | popularity_norm | MinMax |

**Next:** `05_content_based_recommender.ipynb`